# 01 - Capa Bronze\n\nIngesta del archivo `venta_tiendas.csv` desde Google Drive y escritura en Delta Lake.\n\n**Principio Medallion:** Bronze conserva los datos tal como llegan desde la fuente, incluyendo `fecha_transaccion` con `/`.

# Configuración base

Este notebook está preparado para ejecutarse en Google Colab conectado directamente a **Google Cloud Storage (GCS)** mediante autenticación nativa, reemplazando el uso de almacenamiento local o Google Drive.

Estructura esperada en el Bucket (`gs://data-lake-retail/`):

```text
data-lake-retail/
├── raw/
│   ├── venta_tiendas.csv         <-- (Origen: Archivo CSV crudo leído en este notebook)
│   ├── Maestro_Producto.csv
│   ├── Maestro_Tienda.csv
│   └── venta_ecom.csv
├── bronze/
│   └── venta_tiendas_delta/      <-- (Destino: Datos inmutables guardados en formato Delta)
├── silver/
├── gold/
└── evidencias/

In [1]:
# 1. Autenticación con Google Cloud
from google.colab import auth
auth.authenticate_user()
print("Autenticación con GCP exitosa.")

# 2. Instalación de dependencias para Colab
!pip uninstall -y dataproc-spark-connect opentelemetry-api importlib-metadata pyspark delta-spark > /dev/null
!pip install -q importlib-metadata==8.0.0 pyspark==3.4.1 delta-spark==2.4.0

# 3. Descargar el conector GCS manualmente a la carpeta de PySpark
import pyspark
import os
pyspark_jars_dir = os.path.join(pyspark.__path__[0], "jars")
!wget -q https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar -P {pyspark_jars_dir}
print("Conector GCS descargado correctamente.")

import time
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# 4. Configuración de Spark con soporte Delta y GCS
builder = (
    SparkSession.builder
    .appName("Forus_Fase2_BigData")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    # Indicadores para leer rutas gs://
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

# 5. Definición de Rutas en la Nube
NOMBRE_BUCKET = "data-lake-retail"
RUTA_BASE = f"gs://{NOMBRE_BUCKET}"

RUTA_RAW = f"{RUTA_BASE}/raw"
RUTA_BRONZE = f"{RUTA_BASE}/bronze"
RUTA_SILVER = f"{RUTA_BASE}/silver"
RUTA_GOLD = f"{RUTA_BASE}/gold"

print("Ruta base configurada:", RUTA_BASE)

# 6. Lectura del archivo RAW (CSV)
ruta_csv = f"{RUTA_RAW}/venta_tiendas.csv"

inicio = time.time()

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)  # Excelente decisión: Bronze preserva valores originales
    .option("encoding", "UTF-8")
    .csv(ruta_csv)
)

tiempo_lectura = time.time() - inicio

print("Registros RAW:", df_raw.count())
print("Tiempo lectura RAW:", round(tiempo_lectura, 2), "segundos")
df_raw.printSchema()
df_raw.show(5, truncate=False)

# 7. Escritura en formato Delta Lake (Capa Bronze)
# Aquí tomamos la data cruda y la guardamos en la carpeta Bronze del bucket
ruta_destino_bronze = f"{RUTA_BRONZE}/venta_tiendas_delta"

print(f"Guardando datos inmutables en: {ruta_destino_bronze}...")
inicio_escritura = time.time()

df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_destino_bronze)

tiempo_escritura = time.time() - inicio_escritura
print(f"¡Guardado exitoso en Delta Lake! Tiempo: {round(tiempo_escritura, 2)} segundos")

Autenticación con GCP exitosa.
Conector GCS descargado correctamente.
Spark version: 3.4.1
Ruta base configurada: gs://data-lake-retail
Registros RAW: 2250970
Tiempo lectura RAW: 11.13 segundos
root
 |-- id_canal: string (nullable = true)
 |-- numero_transaccion: string (nullable = true)
 |-- numero_pos: string (nullable = true)
 |-- numero_boleta: string (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: string (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- unidades: string (nullable = true)
 |-- venta: string (nullable = true)
 |-- costo: string (nullable = true)

+--------+------------------+----------+-------------+-------------------------+----------------------+--------------+-----------+--------+-----+-----+
|id_canal|numero_transaccion|numero_pos|numero_boleta|fecha_transaccion        |cod_tienda_facturacion|tipo_documento|id_producto|unidades|venta|costo|
+--------+---

In [3]:
# Listar usando el nuevo comando estándar de Google Cloud
print(f"Contenido de la carpeta RAW en GCS ({RUTA_RAW}):")
!gcloud storage ls {RUTA_RAW}

print("\nContenido de la carpeta BRONZE en GCS:")
!gcloud storage ls {RUTA_BRONZE}

Contenido de la carpeta RAW en GCS (gs://data-lake-retail/raw):
gs://data-lake-retail/raw/
gs://data-lake-retail/raw/Maestro_Producto.csv
gs://data-lake-retail/raw/Maestro_Tienda.csv
gs://data-lake-retail/raw/venta_ecom.csv
gs://data-lake-retail/raw/venta_tiendas.csv

Contenido de la carpeta BRONZE en GCS:
gs://data-lake-retail/bronze/
gs://data-lake-retail/bronze/venta_tiendas_delta/


## 2. Validación Bronze

In [4]:
# 8. Validación Bronze (Lectura desde la nube)
ruta_destino_bronze = f"{RUTA_BRONZE}/venta_tiendas_delta"
print(f"Validando escritura desde: {ruta_destino_bronze}")

df_bronze = spark.read.format("delta").load(ruta_destino_bronze)

print("Registros guardados en Bronze:", df_bronze.count())
df_bronze.select("fecha_transaccion").show(5, truncate=False)

Validando escritura desde: gs://data-lake-retail/bronze/venta_tiendas_delta
Registros guardados en Bronze: 2250970
+-------------------------+
|fecha_transaccion        |
+-------------------------+
|31/01/2017 12:00:00 AM CL|
|31/01/2017 12:00:00 AM CL|
|31/01/2017 12:00:00 AM CL|
|31/01/2017 12:00:00 AM CL|
|31/01/2017 12:00:00 AM CL|
+-------------------------+
only showing top 5 rows



## 4. Evidencia de linaje

In [5]:
# 9. Evidencia de Linaje de Datos (Data Lineage)
linaje_bronze = {
    "dataset": "venta_tiendas",
    "origen": ruta_csv,  # Esto ahora apunta a tu gs://.../raw/venta_tiendas.csv
    "destino": ruta_destino_bronze, # Esto apunta a gs://.../bronze/venta_tiendas_delta
    "capa": "Bronze",
    "formato": "Delta Lake",
    "transformaciones": "Sin transformaciones; preserva datos originales",
    "fecha_original": "fecha_transaccion conservada con slash y sufijo AM/PM CL"
}

print("--- DOCUMENTACIÓN DE LINAJE (GOBIERNO DE DATOS) ---")
for k, v in linaje_bronze.items():
    print(f"{k.upper()}: {v}")

--- DOCUMENTACIÓN DE LINAJE (GOBIERNO DE DATOS) ---
DATASET: venta_tiendas
ORIGEN: gs://data-lake-retail/raw/venta_tiendas.csv
DESTINO: gs://data-lake-retail/bronze/venta_tiendas_delta
CAPA: Bronze
FORMATO: Delta Lake
TRANSFORMACIONES: Sin transformaciones; preserva datos originales
FECHA_ORIGINAL: fecha_transaccion conservada con slash y sufijo AM/PM CL
